# 🔻 Stop Resources — cost control

Stops the **Databricks App** and the **Lakebase instance** so they stop billing.
Run it **manually anytime**, or let the scheduled *auto-shutdown* job (created in
Notebook 7, Step 7) run it for you after a few hours.

Restart later from the UI, or by re-running Notebooks 5 (Lakebase) and 7 (App).

Set the widgets to match the app/instance names you used.

In [ ]:
%pip install --quiet --upgrade databricks-sdk
dbutils.library.restartPython()

In [ ]:
import re
dbutils.widgets.text("app_name", "", "Databricks App name (blank = auto per-user)")
dbutils.widgets.text("lakebase_instance", "abi-hackathon-lakebase", "Lakebase instance (shared)")
# When run by the auto-shutdown job, app_name is passed in; for manual runs we
# derive your per-user app name from your identity.
_user = spark.sql("SELECT current_user()").collect()[0][0]
_dslug = re.sub(r"[^a-z0-9]+", "-", _user.split("@")[0].lower()).strip("-")[:30]
APP = dbutils.widgets.get("app_name").strip() or f"abi-genie-app-{_dslug}"
INSTANCE = dbutils.widgets.get("lakebase_instance").strip()

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.database import DatabaseInstance
w = WorkspaceClient()

# Stop the app (no-op if it doesn't exist yet / already stopped).
if APP:
    try:
        w.apps.stop(name=APP)
        print(f"✔ Requested stop of app '{APP}'.")
    except Exception as e:
        print(f"App stop skipped/failed: {e}")

# Stop the Lakebase instance (data is preserved; only compute stops).
if INSTANCE:
    try:
        w.database.update_database_instance(
            name=INSTANCE,
            database_instance=DatabaseInstance(name=INSTANCE, stopped=True),
            update_mask="stopped",
        )
        print(f"✔ Requested stop of Lakebase '{INSTANCE}'.")
    except Exception as e:
        print(f"Lakebase stop skipped/failed: {e}")

print("Done.")